In [25]:
# Basic Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans

# Evaluation
from sklearn.metrics import silhouette_score

In [26]:
#changes
# Load the data
df = pd.read_csv("/content/drive/MyDrive/MachineLearningModels/Unsupervised Learning/data.csv", encoding='latin1')

# First 5 rows
display(df.head())

# Number of rows and columns
print("Shape:", df.shape)

# Column information
df.info()

# Statistical summary
display(df.describe())

# Missing values
display(df.isnull().sum())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


Shape: (541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


What we found

Our dataset has:

541,909 rows
8 columns
Description has 1,454 missing values
CustomerID has 135,080 missing values
InvoiceDate is currently stored as text

Data Preprocessing

Now we clean the data before doing business analysis.
Step 1 — Convert date

In [27]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

Step 2 — Handle missing descriptions
As We don't want missing product names.

In [28]:
df["Description"] = df["Description"].fillna("Unknown")
display(df.isnull().sum())

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


Step 3 — Identify cancelled orders

Some invoices start with "C".

In [29]:
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C")
display(df[df['IsCancelled']].head(10))

print("Cancelled transactions:", df["IsCancelled"].sum())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,True
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom,True
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,True
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,True
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,True
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,True
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom,True
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,True
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,True
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom,True


Cancelled transactions: 9288


In [30]:
#Remove invalid sales transactions

#For revenue analysis, we only want actual positive sales.

sales_df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()
#This removes transactions with:negative quantity,zero quantity,zero/negative price
display(sales_df.head())
print("Shape:", sales_df.shape)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False


Shape: (530104, 9)


In [31]:
#Step 5 — Create Revenue

#This is one of the most important calculations.

sales_df["Revenue"] = (
    sales_df["Quantity"] *
    sales_df["UnitPrice"]
)

##The formula is:

#Revenue = Quantity × Unit Price

In [32]:
#calculate the important business numbers.
total_revenue = sales_df["Revenue"].sum()

total_orders = sales_df["InvoiceNo"].nunique()

total_customers = sales_df["CustomerID"].nunique()

average_order_value = (
    sales_df.groupby("InvoiceNo")["Revenue"].sum().mean()
)

print("Total Revenue: $", round(total_revenue, 2))
print("Total Orders:", total_orders)
print("Unique Customers:", total_customers)
print("Average Order Value: $", round(average_order_value, 2))


Total Revenue: $ 10666684.54
Total Orders: 19960
Unique Customers: 4338
Average Order Value: $ 534.4


Graph 1 — Monthly Revenue Trend

In [ ]:
sales_df["YearMonth"] = (
    sales_df["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

monthly_revenue = (
    sales_df.groupby("YearMonth")["Revenue"]
    .sum()
)

plt.figure(figsize=(10, 5))

plt.plot(
    monthly_revenue.index,
    monthly_revenue.values,
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue ($)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Revenue becomes much stronger toward the end of 2011, with November being the strongest month.

This immediately tells management that the business has significant seasonal growth toward the holiday period.

In [ ]:
#Graph 2 — Top 10 Countries by Revenue

top_countries = (
    sales_df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
display(top_countries)

plt.figure(figsize=(10, 6))

plt.barh(
    top_countries.index,
    top_countries.values
)

plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue ($)")
plt.ylabel("Country")

plt.tight_layout()
plt.show()

The United Kingdom dominates revenue, generating approximately $9.03M of the $10.67M total.

That means the company is highly dependent on its UK market.



In [ ]:
#Graph 3 — Top 10 Products by Revenue
top_products = (
    sales_df.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)
display(top_products)

plt.figure(figsize=(10, 6))

plt.barh(
    top_products.index,
    top_products.values
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue ($)")
plt.ylabel("Product")

plt.tight_layout()
plt.show()

The highest-revenue products include:

DOTCOM POSTAGE
REGENCY CAKESTAND 3 TIER
PAPER CRAFT, LITTLE BIRDIE
WHITE HANGING HEART T-LIGHT HOLDER
PARTY BUNTING

This helps management identify products that contribute strongly to revenue.

In [ ]:
#Graph 4 — Monthly Number of Orders
monthly_orders = (
    sales_df.groupby("YearMonth")["InvoiceNo"]
    .nunique()
)
display(monthly_orders)
plt.figure(figsize=(10, 5))

plt.plot(
    monthly_orders.index,
    monthly_orders.values,
    marker="o"
)

plt.title("Monthly Number of Orders")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

The number of orders rises significantly during September–November, with November having the highest order volume.

This supports the revenue trend and suggests a strong seasonal demand pattern.

In [ ]:
# Set the latest date in the dataset as the reference date
snapshot_date = sales_df["InvoiceDate"].max() + pd.Timedelta(days=1)

# Calculate Recency, Frequency, and Monetary for each customer
rfm = sales_df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()
print(rfm)

# Display the first 5 customers
display(rfm.head())

🔹 RFM Feature Engineering (Detailed Notes)

Recency, Frequency, and Monetary (RFM) is a behavioral segmentation technique used to quantify customer purchasing patterns.

* Recency measures how recently a customer made a purchase. Lower values indicate more recent activity.

* Frequency counts how many unique transactions a customer made.

* Monetary represents total spending by the customer.

By aggregating transactional data into customer-level metrics, we transform raw invoice data into structured behavioral features suitable for clustering.

Customers with negative monetary values (refunds) were removed to prevent distortion in cluster formation.